# 🚀 Capstone Projects

Apply everything you've learned. Each project builds on previous phases.

| Project | Phases Used | Difficulty |
|---------|------------|------------|
| P1: MNIST Classifier | 1–5 | ⭐ |
| P2: CIFAR-10 Transfer Learning | 7 | ⭐⭐ |
| P3: Sentiment Analysis (LSTM) | 8 | ⭐⭐ |
| P4: Text Generation (Transformer) | 9 | ⭐⭐⭐ |
| P5: GAN Image Generation | 11 | ⭐⭐⭐ |
| P6: Deploy to API | 13 | ⭐⭐ |

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## Project 1: MNIST Digit Classifier

**Goal**: Build a complete training pipeline from scratch.  
**Target**: >98% test accuracy.

### Tasks:
1. Load MNIST with transforms
2. Split into train/val/test
3. Build a model (FC or CNN)
4. Train with validation monitoring
5. Evaluate on test set
6. Save the best model

In [ ]:
# --- YOUR CODE: Data ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

train_set, val_set = random_split(full_train, [50000, 10000])
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)

In [ ]:
# --- YOUR CODE: Model ---
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define your layers
        pass

    def forward(self, x):
        # TODO: Define forward pass
        pass

In [ ]:
# --- YOUR CODE: Training Loop ---
# model = MNISTModel().to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(model.parameters(), lr=1e-3)
#
# TODO: Implement training with validation, early stopping, save best model

---
## Project 2: CIFAR-10 with Transfer Learning

**Goal**: Use a pretrained ResNet to classify CIFAR-10 images.  
**Target**: >90% test accuracy.

### Tasks:
1. Load CIFAR-10 with augmentation
2. Load pretrained ResNet18
3. Replace final layer for 10 classes
4. Try both feature extraction and fine-tuning
5. Compare results

In [ ]:
# --- YOUR CODE ---
# Hint: See Phase 7 notebook for the pattern
#
# from torchvision.models import resnet18, ResNet18_Weights
# model = resnet18(weights=ResNet18_Weights.DEFAULT)
# model.fc = nn.Linear(model.fc.in_features, 10)
#
# TODO: Implement data loading, training, comparison

---
## Project 3: Sentiment Analysis with LSTM

**Goal**: Classify movie reviews as positive/negative.  
**Dataset**: Use torchtext or load IMDB manually.

### Tasks:
1. Tokenize text and build vocabulary
2. Create Dataset with padding
3. Build Embedding + BiLSTM + FC model
4. Train and evaluate
5. Test on custom sentences

In [ ]:
# --- YOUR CODE ---
# Hint: See Phase 8 TextClassifier
#
# class SentimentModel(nn.Module):
#     def __init__(self, vocab_size, embed_dim, hidden_dim):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim)
#         self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
#         self.fc = nn.Linear(hidden_dim * 2, 1)
#
#     def forward(self, x):
#         ...
#
# TODO: Implement full pipeline

---
## Project 4: Text Generation with Transformer

**Goal**: Build a character-level or word-level text generator.  
**Architecture**: Decoder-only transformer (mini GPT).

### Tasks:
1. Prepare text dataset (Shakespeare, Wikipedia, etc.)
2. Build tokenizer (character-level is simplest)
3. Implement MiniGPT from Phase 9
4. Train with teacher forcing
5. Generate text autoregressively

In [ ]:
# --- YOUR CODE ---
# Hint: See Phase 9 MiniGPT
#
# Steps:
# 1. text = open('input.txt').read()
# 2. chars = sorted(set(text))
# 3. char_to_idx = {c: i for i, c in enumerate(chars)}
# 4. Encode text → tensor of indices
# 5. Create sliding window dataset
# 6. Train MiniGPT
# 7. Generate: start with prompt, predict next token, append, repeat
#
# TODO: Implement full pipeline

---
## Project 5: GAN Image Generation

**Goal**: Generate realistic handwritten digits with a DCGAN.  
**Dataset**: MNIST or Fashion-MNIST.

### Tasks:
1. Build convolutional Generator (ConvTranspose2d)
2. Build convolutional Discriminator (Conv2d)
3. Train with adversarial loss
4. Generate and visualize samples
5. Plot training loss curves

In [ ]:
# --- YOUR CODE ---
# Hint: See Phase 11 GAN section
#
# DCGAN tips:
# - Generator: Linear → Reshape → ConvTranspose2d (upsample)
# - Discriminator: Conv2d (downsample) → Flatten → Linear
# - Use BatchNorm in both (except D's first and G's last layer)
# - LeakyReLU in D, ReLU in G
# - Adam with lr=2e-4, betas=(0.5, 0.999)
#
# TODO: Implement full pipeline

---
## Project 6: Deploy Model to API

**Goal**: Serve your MNIST model as a REST API.  
**Options**: FastAPI (local) or AWS SageMaker (cloud).

### Tasks:
1. Export trained model (TorchScript or ONNX)
2. Build FastAPI endpoint
3. Accept image input, return prediction
4. (Bonus) Deploy to SageMaker endpoint

In [ ]:
# --- FastAPI Template ---
fastapi_code = '''
# app.py — run with: uvicorn app:app --reload
from fastapi import FastAPI, UploadFile
import torch
import torchvision.transforms as transforms
from PIL import Image
import io

app = FastAPI()

# Load model once at startup
model = torch.jit.load("production_model.pt")
model.eval()

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

@app.post("/predict")
async def predict(file: UploadFile):
    image = Image.open(io.BytesIO(await file.read()))
    tensor = transform(image).view(1, 784)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax().item()
        confidence = probs.max().item()

    return {"prediction": pred, "confidence": f"{confidence:.2%}"}
'''
print(fastapi_code)

---
## 🎯 Tips for All Projects

1. **Start simple** — get a baseline working first, then improve
2. **Overfit one batch** — verify your model can learn before full training
3. **Track metrics** — log train/val loss and accuracy every epoch
4. **Save checkpoints** — don't lose hours of training to a crash
5. **Visualize** — plot losses, sample predictions, confusion matrices
6. **Iterate** — try different architectures, hyperparameters, augmentations

**Good luck! 🚀**